# 02 KPI framework and store performance

Here I move from cleaned data to business KPIs. The main goal is not just to rank stores by activation, but to compare stores using volume, revenue/profit, upselling, productivity, and product/service mix.

I kept a few validation-style checks in the notebook because KPI definitions can easily get mixed up.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 1) Import libraries

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path

pd.set_option("display.max_columns", 120)
pd.set_option("display.width", 160)

In [ ]:
# 2) File paths

BASE_DIR = Path("/content/drive/MyDrive/Colab Notebooks/Practicum/Demo File")

PROCESSED_DIR = BASE_DIR / "data" / "processed"
VISUAL_DIR = BASE_DIR / "visuals"

VISUAL_DIR.mkdir(parents=True, exist_ok=True)

clean_daily_path = PROCESSED_DIR / "clean_daily_master.csv"
clean_monthly_path = PROCESSED_DIR / "clean_monthly_kpi.csv"
kpi_dictionary_path = PROCESSED_DIR / "kpi_dictionary.csv"

print("Clean daily file exists:", clean_daily_path.exists())
print("Clean monthly file exists:", clean_monthly_path.exists())
print("KPI dictionary exists:", kpi_dictionary_path.exists())

In [ ]:
# 3) Loading clean data

daily = pd.read_csv(clean_daily_path)
monthly = pd.read_csv(clean_monthly_path)
kpi_dictionary = pd.read_csv(kpi_dictionary_path)

daily["Date"] = pd.to_datetime(daily["Date"])
monthly["Month"] = pd.to_datetime(monthly["Month"])

print("Daily shape:", daily.shape)
print("Monthly KPI shape:", monthly.shape)

display(daily.head())
display(monthly.head())
display(kpi_dictionary.head())

In [ ]:
# 4) Quick check before analysis

# I am doing this quick check again because mistakes in date/store IDs can
# quietly affect all store-level summaries.

print("Daily date range:", daily["Date"].min(), "to", daily["Date"].max())
print("Monthly date range:", monthly["Month"].min(), "to", monthly["Month"].max())

print("Stores in daily data:", daily["Store_ID"].nunique())
print("Stores in monthly data:", monthly["Store_ID"].nunique())

print("Duplicate store-date rows:", daily.duplicated(subset=["Date", "Store_ID"]).sum())
print("Duplicate store-month rows:", monthly.duplicated(subset=["Month", "Store_ID"]).sum())

In [ ]:
# 5) Create KPI framework table

# This table is not a statistical model. It is a business measurement framework.
# It helps organize which metrics are used for which type of question.

kpi_framework = pd.DataFrame({
    "KPI_Category": [
        "Activity / Volume",
        "Revenue / Profit",
        "Upselling Efficiency",
        "Operational Efficiency",
        "Product / Service Mix"
    ],
    "Metrics": [
        "Total Activation, New Activation, PPD, Boxes",
        "Account Gross, Accessory Profit, Acc_Dollar, Base MRC, Feature Rev",
        "Acc/PPD, Acc/Box, Accessory Profit per Activation",
        "PPD per Hour, Boxes per Hour, Conversion %",
        "AAL, Tablet, HINT, Watch, Trade"
    ],
    "Business_Question": [
        "How much business activity is each store generating?",
        "Which stores are producing stronger financial outcomes?",
        "How well are stores converting transactions into accessory value?",
        "How productive are stores relative to operating hours or customer interactions?",
        "Which products or services are contributing to store activity?"
    ]
})

display(kpi_framework)

In [ ]:
# saving this output so the next notebook / Power BI can reuse the same table
# 6) Save KPI framework

kpi_framework_path = PROCESSED_DIR / "kpi_framework.csv"
kpi_framework.to_csv(kpi_framework_path, index=False)

print("Saved KPI framework:", kpi_framework_path)

In [ ]:
# 7) Aggregate daily data to store-month level

# The daily file has activation/profit fields, while the KPI file is monthly.
# To compare them, I need both datasets at the same store-month level.

daily["Month"] = daily["Date"].dt.to_period("M").dt.to_timestamp()

monthly_from_daily = (
    daily
    .groupby(
        ["Month", "Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"],
        as_index=False
    )
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        New_Activation=("New_Activation", "sum"),
        Upgrade_SOR=("Upgrade_SOR", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Promotion_Days_Daily=("Promotion_Flag", "sum"),
        Local_Event_Days_Daily=("Local_Event_Flag", "sum"),
        Inventory_Issue_Days_Daily=("Inventory_Issue_Flag", "sum")
    )
)

print("Monthly summary from daily data:", monthly_from_daily.shape)
display(monthly_from_daily.head())

In [ ]:
# 8) Merge monthly KPI data with monthly daily-summary data

# This creates the main dataset for KPI relationship analysis.
# It lets me compare daily report outcomes with monthly KPI fields.

monthly_analysis = monthly_from_daily.merge(
    monthly,
    on=["Month", "Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"],
    how="left",
    suffixes=("", "_KPI")
)

print("Monthly analysis shape:", monthly_analysis.shape)
display(monthly_analysis.head())

In [ ]:
# 9) Check merge quality

# If important KPI fields are missing after the merge, something is wrong
# with the store/month keys.

important_cols = ["PPD", "Boxes", "Hours", "Acc_Dollar", "AAL", "Tablet", "HINT"]

print("Missing values after merge:")
display(monthly_analysis[important_cols].isna().sum())

In [ ]:
# 10) Create derived KPI metrics

# These metrics help compare stores fairly.
# For example, total accessory profit is useful, but accessory profit per activation
# tells us whether the store is efficient relative to activity volume.

monthly_analysis["PPD_per_Hour"] = monthly_analysis["PPD"] / monthly_analysis["Hours"]
monthly_analysis["Boxes_per_Hour"] = monthly_analysis["Boxes"] / monthly_analysis["Hours"]

monthly_analysis["Accessory_Revenue_per_PPD"] = monthly_analysis["Acc_Dollar"] / monthly_analysis["PPD"]
monthly_analysis["Accessory_Revenue_per_Box"] = monthly_analysis["Acc_Dollar"] / monthly_analysis["Boxes"]

monthly_analysis["Accessory_Profit_per_Activation"] = (
    monthly_analysis["Accessory_Profit"] / monthly_analysis["Total_Activation"]
)

monthly_analysis["Account_Gross_per_Activation"] = (
    monthly_analysis["Account_Gross"] / monthly_analysis["Total_Activation"]
)

# Replace infinite values if denominator was zero.
monthly_analysis = monthly_analysis.replace([np.inf, -np.inf], np.nan)

display(
    monthly_analysis[
        [
            "Month", "Store_ID", "Total_Activation", "PPD", "Hours",
            "PPD_per_Hour", "Accessory_Revenue_per_PPD",
            "Accessory_Revenue_per_Box", "Accessory_Profit_per_Activation",
            "Account_Gross_per_Activation"
        ]
    ].head()
)

In [ ]:
# 11) Create store-level performance summary

store_performance = (
    monthly_analysis
    .groupby(["Store_ID", "Store_Name", "Market_Zone", "Store_Type", "Store_Size"], as_index=False)
    .agg(
        Total_Activation=("Total_Activation", "sum"),
        New_Activation=("New_Activation", "sum"),
        Upgrade_SOR=("Upgrade_SOR", "sum"),
        PPD=("PPD", "sum"),
        Boxes=("Boxes", "sum"),
        Hours=("Hours", "sum"),
        Account_Gross=("Account_Gross", "sum"),
        Accessory_Profit=("Accessory_Profit", "sum"),
        Acc_Dollar=("Acc_Dollar", "sum"),
        Acc_Qty=("Acc_Qty", "sum"),
        AAL=("AAL", "sum"),
        Tablet=("Tablet", "sum"),
        HINT=("HINT", "sum"),
        Watch=("Watch", "sum"),
        Trade=("Trade", "sum"),
        Promotion_Days=("Promotion_Days", "sum"),
        Inventory_Issue_Days=("Inventory_Issue_Days", "sum"),
        Local_Event_Days=("Local_Event_Days", "sum")
    )
)

# Add store-level efficiency metrics
store_performance["PPD_per_Hour"] = store_performance["PPD"] / store_performance["Hours"]
store_performance["Boxes_per_Hour"] = store_performance["Boxes"] / store_performance["Hours"]
store_performance["Accessory_Revenue_per_PPD"] = store_performance["Acc_Dollar"] / store_performance["PPD"]
store_performance["Accessory_Revenue_per_Box"] = store_performance["Acc_Dollar"] / store_performance["Boxes"]
store_performance["Accessory_Profit_per_Activation"] = store_performance["Accessory_Profit"] / store_performance["Total_Activation"]
store_performance["Account_Gross_per_Activation"] = store_performance["Account_Gross"] / store_performance["Total_Activation"]

store_performance = store_performance.replace([np.inf, -np.inf], np.nan)

display(store_performance.sort_values("Total_Activation", ascending=False).head(10))

In [ ]:
# 12) Save store performance summary

store_performance_path = PROCESSED_DIR / "store_performance_summary.csv"
monthly_analysis_path = PROCESSED_DIR / "monthly_analysis_dataset.csv"

store_performance.to_csv(store_performance_path, index=False)
monthly_analysis.to_csv(monthly_analysis_path, index=False)

print("Saved store performance summary:", store_performance_path)
print("Saved monthly analysis dataset:", monthly_analysis_path)

In [ ]:
# 13) Total Activation by store

plot_df = store_performance.sort_values("Total_Activation", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["Total_Activation"])

plt.title("Total Activation by Store")
plt.xlabel("Total Activation")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "kpi_total_activation_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 14) Account Gross by store

plot_df = store_performance.sort_values("Account_Gross", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["Account_Gross"])

plt.title("Account Gross by Store")
plt.xlabel("Account Gross")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "kpi_account_gross_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 15) Accessory Profit by store

plot_df = store_performance.sort_values("Accessory_Profit", ascending=True)

plt.figure(figsize=(10, 8))
plt.barh(plot_df["Store_ID"], plot_df["Accessory_Profit"])

plt.title("Accessory Profit by Store")
plt.xlabel("Accessory Profit")
plt.ylabel("Store ID")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "kpi_accessory_profit_by_store.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 16) Product and service activity summary

# These are count-based fields. I am not treating them as revenue.
# They help show what kind of activity each store has.

product_service_cols = ["AAL", "Tablet", "HINT", "Watch", "Trade"]

product_service_summary = store_performance[
    ["Store_ID", "Store_Name"] + product_service_cols
].copy()

display(product_service_summary.head())

In [ ]:
# 17) Total product/service activity by category

product_totals = (
    product_service_summary[product_service_cols]
    .sum()
    .reset_index()
)

product_totals.columns = ["Product_Service", "Total_Count"]
product_totals = product_totals.sort_values("Total_Count", ascending=True)

plt.figure(figsize=(8, 5))
plt.barh(product_totals["Product_Service"], product_totals["Total_Count"])

plt.title("Product and Service Activity by Category")
plt.xlabel("Total Count")
plt.ylabel("Product / Service")
plt.grid(axis="x")
plt.tight_layout()

plt.savefig(VISUAL_DIR / "product_service_activity_totals.png", dpi=300, bbox_inches="tight")
plt.show()

display(product_totals.sort_values("Total_Count", ascending=False))

In [ ]:
# correlation is useful here but I am only treating it as association
# 18) KPI relationship variables

# I am selecting only variables that have useful business meaning.
# Some obvious relationships may be very high, but not always useful to discuss.

relationship_vars = [
    "Total_Activation",
    "New_Activation",
    "PPD",
    "Boxes",
    "Hours",
    "QPAY",
    "Account_Gross",
    "Accessory_Profit",
    "Acc_Dollar",
    "Acc_Qty",
    "AAL",
    "Tablet",
    "HINT",
    "Watch",
    "Trade",
    "Promotion_Days",
    "Inventory_Issue_Days",
    "Local_Event_Days",
    "PPD_per_Hour",
    "Accessory_Revenue_per_PPD",
    "Accessory_Revenue_per_Box",
    "Accessory_Profit_per_Activation",
    "Account_Gross_per_Activation"
]

relationship_df = monthly_analysis[relationship_vars].copy()

corr_matrix = relationship_df.corr(numeric_only=True)

display(corr_matrix.round(2))

In [ ]:
# 19) Correlations with Account Gross

account_gross_corr = (
    corr_matrix["Account_Gross"]
    .drop("Account_Gross")
    .sort_values(ascending=False)
    .reset_index()
)

account_gross_corr.columns = ["Variable", "Correlation_with_Account_Gross"]

display(account_gross_corr.head(15))

In [ ]:
# 20) Correlations with Accessory Profit

accessory_profit_corr = (
    corr_matrix["Accessory_Profit"]
    .drop("Accessory_Profit")
    .sort_values(ascending=False)
    .reset_index()
)

accessory_profit_corr.columns = ["Variable", "Correlation_with_Accessory_Profit"]

display(accessory_profit_corr.head(15))

In [ ]:
# 21) Selected KPI relationships

# I am manually selecting the relationships that are easier to explain
# and useful for business monitoring. This avoids reporting random correlations.

selected_relationships = [
    ("PPD", "Account_Gross", "Does broader activity relate to account gross?"),
    ("AAL", "Account_Gross", "Is add-a-line activity connected with account gross?"),
    ("Tablet", "Account_Gross", "Is tablet activity useful as a service indicator?"),
    ("HINT", "Account_Gross", "Is home internet activity related to account gross?"),
    ("Accessory_Revenue_per_Box", "Accessory_Profit_per_Activation", "Does accessory revenue per device relate to profit efficiency?"),
    ("Promotion_Days", "Total_Activation", "Do promotion days relate to activation volume?"),
    ("Inventory_Issue_Days", "Total_Activation", "Do inventory issue days relate to activation volume?")
]

relationship_summary_rows = []

for x_var, y_var, question in selected_relationships:
    corr_value = monthly_analysis[[x_var, y_var]].corr().iloc[0, 1]

    relationship_summary_rows.append({
        "X_Variable": x_var,
        "Y_Variable": y_var,
        "Business_Question": question,
        "Correlation": round(corr_value, 3)
    })

kpi_relationship_summary = pd.DataFrame(relationship_summary_rows)

display(kpi_relationship_summary)

In [ ]:
# 22) Add interpretation labels

def label_correlation(value):
    abs_value = abs(value)

    if abs_value >= 0.70:
        return "Strong"
    elif abs_value >= 0.40:
        return "Moderate"
    elif abs_value >= 0.20:
        return "Weak"
    else:
        return "Very weak"

kpi_relationship_summary["Relationship_Strength"] = kpi_relationship_summary["Correlation"].apply(label_correlation)

display(kpi_relationship_summary)

In [ ]:
# 23) Scatterplot: PPD vs Account Gross

plt.figure(figsize=(8, 6))
plt.scatter(monthly_analysis["PPD"], monthly_analysis["Account_Gross"], alpha=0.6)

plt.title("PPD vs Account Gross")
plt.xlabel("PPD")
plt.ylabel("Account Gross")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "relationship_ppd_account_gross.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 24) Scatterplot: Accessory Revenue per Box vs Accessory Profit per Activation

plt.figure(figsize=(8, 6))
plt.scatter(
    monthly_analysis["Accessory_Revenue_per_Box"],
    monthly_analysis["Accessory_Profit_per_Activation"],
    alpha=0.6
)

plt.title("Accessory Revenue per Box vs Accessory Profit per Activation")
plt.xlabel("Accessory Revenue per Box")
plt.ylabel("Accessory Profit per Activation")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "relationship_acc_box_profit_efficiency.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 25) Scatterplot: AAL vs Account Gross

plt.figure(figsize=(8, 6))
plt.scatter(monthly_analysis["AAL"], monthly_analysis["Account_Gross"], alpha=0.6)

plt.title("AAL vs Account Gross")
plt.xlabel("AAL")
plt.ylabel("Account Gross")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "relationship_aal_account_gross.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 26) Scatterplot: Inventory Issue Days vs Total Activation

plt.figure(figsize=(8, 6))
plt.scatter(
    monthly_analysis["Inventory_Issue_Days"],
    monthly_analysis["Total_Activation"],
    alpha=0.6
)

plt.title("Inventory Issue Days vs Total Activation")
plt.xlabel("Inventory Issue Days")
plt.ylabel("Total Activation")
plt.grid(True)
plt.tight_layout()

plt.savefig(VISUAL_DIR / "relationship_inventory_activation.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
# 27) Save KPI relationship outputs

corr_matrix_path = PROCESSED_DIR / "kpi_correlation_matrix.csv"
kpi_relationship_summary_path = PROCESSED_DIR / "kpi_relationship_summary.csv"

corr_matrix.to_csv(corr_matrix_path)
kpi_relationship_summary.to_csv(kpi_relationship_summary_path, index=False)

print("Saved correlation matrix:", corr_matrix_path)
print("Saved KPI relationship summary:", kpi_relationship_summary_path)

In [ ]:
# 28) Final notebook summary

top_activation_store = store_performance.sort_values("Total_Activation", ascending=False).iloc[0]
top_account_gross_store = store_performance.sort_values("Account_Gross", ascending=False).iloc[0]
top_accessory_profit_store = store_performance.sort_values("Accessory_Profit", ascending=False).iloc[0]

print("KPI and Store Performance Summary")
print("---------------------------------")
print("Stores analyzed:", store_performance["Store_ID"].nunique())
print("Store-month records analyzed:", len(monthly_analysis))
print()
print("Highest activation store:", top_activation_store["Store_ID"], "-", top_activation_store["Store_Name"])
print("Highest account gross store:", top_account_gross_store["Store_ID"], "-", top_account_gross_store["Store_Name"])
print("Highest accessory profit store:", top_accessory_profit_store["Store_ID"], "-", top_accessory_profit_store["Store_Name"])
print()
print("Selected KPI relationships:")
display(kpi_relationship_summary)